# Fundamentals 00.4 - Runtime vLLM Provider API

Objetivo: probar `vllm-runtime` como provider OpenAI-compatible para Agentic Systems, con una ruta all-in-one que puede levantar un servidor vLLM en Colab/GPU cuando se requiere.

Este tutorial separa tres responsabilidades:

```text
Unsloth / vLLM infra opcional  ->  endpoint OpenAI-compatible  ->  Agentic Systems runtime(provider="vllm-runtime")
```

Regla de dise?o:

- Agentic Systems no administra CUDA ni entrena modelos.
- vLLM expone el endpoint OpenAI-compatible.
- Unsloth queda como ruta de carga r?pida, 4-bit y futuro fine-tuning/export.
- El notebook usa gates claros: si `/v1/models` no responde, no ejecuta inferencia.


## Parámetros de `vllm-runtime`

Los flags del notebook controlan infraestructura opcional; los parametros `VLLM_*` describen el endpoint OpenAI-compatible que consume Agentic Systems.

| Parametro | Que controla |
|---|---|
| `RUN_INSTALL` | Instalacion opcional de dependencias. |
| `RUN_UNSLOTH_4BIT_SMOKE` | Smoke local opcional con carga 4-bit. |
| `RUN_VLLM_SERVER_SETUP` | Arranque opcional del servidor vLLM. |
| `RUN_OPENAI_SDK_SMOKE` | Prueba directa del endpoint con el cliente OpenAI. |
| `RUN_AGENTIC_SYSTEMS_SMOKE` | Prueba del mismo endpoint mediante Agentic Systems. |
| `VLLM_MODEL` / `VLLM_MODE` | Modelo servido y perfil de recursos. |
| `VLLM_BASE_URL` / `VLLM_API_KEY` | Direccion y credencial del endpoint compatible. |
| `VLLM_TOOL_CALL_PARSER` / `VLLM_REASONING_PARSER` | Parsers habilitados al levantar vLLM. |

## 0) Flags y configuraci?n del notebook

Cambia esta celda antes de ejecutar. Por default el notebook no instala paquetes ni levanta servidor, para que pueda abrirse en VSCode/local sin romper el ambiente.

Para Colab T4 empieza con:

```python
RUN_INSTALL = True
RUN_VLLM_SERVER_SETUP = True
VLLM_MODE = "FAST"
VLLM_MODEL = "unsloth/Qwen3-0.6B"
```

Para probar modelos m?s grandes, primero valida que `/v1/models` responda con el modelo peque?o.


In [ ]:
import os

# -----------------------------
# Execution flags
# -----------------------------
RUN_INSTALL = True
RUN_UNSLOTH_4BIT_SMOKE = False
RUN_VLLM_SERVER_SETUP = True
RUN_OPENAI_SDK_SMOKE = True
RUN_AGENTIC_SYSTEMS_SMOKE = True

# -----------------------------
# Model candidates
# -----------------------------
VLLM_MODEL_CANDIDATES = [
    "unsloth/Qwen3-0.6B",
    "unsloth/Qwen3-4B-Instruct-2507",
    "unsloth/Qwen3-4B-Thinking-2507",
]

VLLM_MODEL = os.getenv("VLLM_MODEL", VLLM_MODEL_CANDIDATES[0])
VLLM_MODE = os.getenv("VLLM_MODE", "FAST")  # FAST, MEDIUM, POWER

# -----------------------------
# Endpoint config
# -----------------------------
VLLM_HOST = os.getenv("VLLM_HOST", "127.0.0.1")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", f"http://{VLLM_HOST}:{VLLM_PORT}/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")

# -----------------------------
# vLLM/Qwen parser config
# -----------------------------
VLLM_ENABLE_TOOL_CALLING = True
VLLM_TOOL_CALL_PARSER = os.getenv("VLLM_TOOL_CALL_PARSER", "hermes")
VLLM_REASONING_PARSER = os.getenv("VLLM_REASONING_PARSER", "qwen3")
VLLM_DISABLE_THINKING_FOR_SDK_SMOKE = os.getenv("VLLM_DISABLE_THINKING_FOR_SDK_SMOKE", "true").lower() in {"1", "true", "yes"}

# -----------------------------
# Unsloth smoke config
# -----------------------------
UNSLOTH_LOAD_IN_4BIT = True
UNSLOTH_FAST_INFERENCE = True
UNSLOTH_MAX_SEQ_LENGTH = 2048

# Normalize env for Agentic Systems and OpenAI SDK.
os.environ["VLLM_MODEL"] = VLLM_MODEL
os.environ["VLLM_BASE_URL"] = VLLM_BASE_URL
os.environ["VLLM_API_KEY"] = VLLM_API_KEY

NOTEBOOK_CONFIG = {
    "RUN_INSTALL": RUN_INSTALL,
    "RUN_UNSLOTH_4BIT_SMOKE": RUN_UNSLOTH_4BIT_SMOKE,
    "RUN_VLLM_SERVER_SETUP": RUN_VLLM_SERVER_SETUP,
    "RUN_OPENAI_SDK_SMOKE": RUN_OPENAI_SDK_SMOKE,
    "RUN_AGENTIC_SYSTEMS_SMOKE": RUN_AGENTIC_SYSTEMS_SMOKE,
    "VLLM_MODEL": VLLM_MODEL,
    "VLLM_MODE": VLLM_MODE,
    "VLLM_BASE_URL": VLLM_BASE_URL,
    "VLLM_API_KEY_configured": bool(VLLM_API_KEY),
    "VLLM_ENABLE_TOOL_CALLING": VLLM_ENABLE_TOOL_CALLING,
    "VLLM_TOOL_CALL_PARSER": VLLM_TOOL_CALL_PARSER,
    "VLLM_REASONING_PARSER": VLLM_REASONING_PARSER,
    "VLLM_DISABLE_THINKING_FOR_SDK_SMOKE": VLLM_DISABLE_THINKING_FOR_SDK_SMOKE,
    "UNSLOTH_LOAD_IN_4BIT": UNSLOTH_LOAD_IN_4BIT,
    "UNSLOTH_FAST_INFERENCE": UNSLOTH_FAST_INFERENCE,
}

NOTEBOOK_CONFIG


## 1) Instalaci?n opcional

`agentic-systems[openai]` instala el cliente OpenAI-compatible que usa `vllm-runtime`.

Para el servidor GPU, la ruta recomendada por Unsloth/vLLM es instalar vLLM con `uv` y `--torch-backend=auto`. Despu?s de instalar en Colab, reinicia el runtime antes de continuar.


In [ ]:
if RUN_INSTALL:
    import subprocess
    import sys

    commands = [
        [sys.executable, "-m", "pip", "install", "-U", "pip"],
        [sys.executable, "-m", "pip", "install", "-U", "agentic-systems[openai]", "unsloth", "uv", "huggingface_hub", "openai"],
        [sys.executable, "-m", "uv", "pip", "install", "-U", "vllm", "--torch-backend=auto"],
    ]

    for cmd in commands:
        print("$", " ".join(cmd))
        subprocess.check_call(cmd)

    print("Instalaci?n completada. En Colab reinicia runtime antes de seguir.")
else:
    print("RUN_INSTALL=False. Se asume que las dependencias ya existen en el kernel activo.")


## 2) Imports y diagn?stico de entorno

La versi?n de `vllm` se consulta con `importlib.metadata` porque `import vllm` puede fallar si la rueda CUDA no corresponde al runtime.


In [ ]:
from __future__ import annotations

import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version as package_version
from urllib.request import Request

import agentic_systems as toolkit


def safe_package_version(name: str) -> str | None:
    try:
        return package_version(name)
    except PackageNotFoundError:
        return None


def environment_snapshot() -> dict:
    payload = {
        "python": sys.executable,
        "agentic_systems": getattr(toolkit, "__version__", "unknown"),
        "vllm_installed": importlib.util.find_spec("vllm") is not None,
        "vllm_version": safe_package_version("vllm"),
        "unsloth_installed": importlib.util.find_spec("unsloth") is not None,
        "unsloth_version": safe_package_version("unsloth"),
        "transformers_version": safe_package_version("transformers"),
        "tokenizers_version": safe_package_version("tokenizers"),
        "torch_version": safe_package_version("torch"),
    }
    try:
        import torch
        payload.update({
            "torch_cuda": torch.version.cuda,
            "cuda_available": torch.cuda.is_available(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        })
    except Exception as exc:
        payload["torch_error"] = str(exc)
    return payload


toolkit.show(NOTEBOOK_CONFIG, title="Notebook config")
toolkit.show(environment_snapshot(), title="Environment snapshot")


## 3) RuntimeConfig de Agentic Systems

`toolkit.runtime(provider="vllm-runtime")` no ejecuta el modelo. Solo declara que Agentic Systems consumir? un endpoint vLLM OpenAI-compatible.


In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=0,
    max_tool_calls=4,
    max_turns=4,
    max_concurrency=1,
)

vllm_runtime = toolkit.runtime(provider="vllm-runtime", scheduler=scheduler)
auto_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)

toolkit.show(vllm_runtime.describe(), title="vLLM runtime - describe")
toolkit.show(auto_runtime.describe(), title="Auto runtime - describe")


## 4) Unsloth 4-bit fast-inference smoke opcional

Esta celda valida la ruta Unsloth de carga r?pida y 4-bit. No es necesaria para Agentic Systems; sirve para preparar el camino de modelos entrenados/exportados con Unsloth.


In [ ]:
if RUN_UNSLOTH_4BIT_SMOKE:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=VLLM_MODEL,
        max_seq_length=UNSLOTH_MAX_SEQ_LENGTH,
        load_in_4bit=UNSLOTH_LOAD_IN_4BIT,
        fast_inference=UNSLOTH_FAST_INFERENCE,
    )
    FastLanguageModel.for_inference(model)
    toolkit.show(
        {
            "status": "ok",
            "model": VLLM_MODEL,
            "load_in_4bit": UNSLOTH_LOAD_IN_4BIT,
            "fast_inference": UNSLOTH_FAST_INFERENCE,
        },
        title="Unsloth 4-bit smoke",
    )
else:
    toolkit.show({"status": "skipped", "reason": "RUN_UNSLOTH_4BIT_SMOKE=False"}, title="Unsloth 4-bit smoke")


## 5) Servidor vLLM opcional

Esta celda levanta el endpoint OpenAI-compatible con `vllm serve`. Usa perfiles conservadores:

| Perfil | Uso |
|---|---|
| `FAST` | T4 seguro, contexto corto. |
| `MEDIUM` | L4/T4 con margen. |
| `POWER` | L4/A100, m?s contexto, menos concurrencia. |


In [ ]:
@dataclass
class VllmServerConfig:
    model: str
    served_model_name: str
    host: str = VLLM_HOST
    port: int = VLLM_PORT
    gpu_memory_utilization: float = 0.40
    max_model_len: int = 2048
    max_num_seqs: int = 4
    tool_call_parser: str = VLLM_TOOL_CALL_PARSER
    reasoning_parser: str = VLLM_REASONING_PARSER
    enable_tool_choice: bool = VLLM_ENABLE_TOOL_CALLING


def vllm_profile(mode: str, model: str) -> VllmServerConfig:
    mode = mode.upper().strip()
    if mode == "POWER":
        return VllmServerConfig(model=model, served_model_name=model, gpu_memory_utilization=0.90, max_model_len=32768, max_num_seqs=1)
    if mode == "MEDIUM":
        return VllmServerConfig(model=model, served_model_name=model, gpu_memory_utilization=0.55, max_model_len=4096, max_num_seqs=4)
    return VllmServerConfig(model=model, served_model_name=model, gpu_memory_utilization=0.40, max_model_len=2048, max_num_seqs=4)


def cleanup_gpu_and_vllm_processes() -> None:
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    os.system('pkill -f "vllm serve" || true')
    os.system('pkill -f "vllm.entrypoints.openai.api_server" || true')


def url_open_no_proxy(url: str, *, timeout: float = 3.0) -> dict:
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    request = Request(url, headers={"Authorization": f"Bearer {VLLM_API_KEY}"})
    with opener.open(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def tail_text(path: str, *, max_chars: int = 8000) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as handle:
            return handle.read()[-max_chars:]
    except FileNotFoundError:
        return ""


def wait_for_models(base_url: str, process: subprocess.Popen | None = None, *, log_path: str = "vllm_server.log", timeout_s: int = 300) -> dict:
    models_url = base_url.rstrip("/") + "/models"
    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        if process is not None and process.poll() is not None:
            return {"status": "failed", "models_url": models_url, "returncode": process.returncode, "log_tail": tail_text(log_path)}
        try:
            payload = url_open_no_proxy(models_url, timeout=3)
            return {"status": "ok", "models_url": models_url, "response": payload}
        except Exception as exc:
            last_error = str(exc)
            time.sleep(3)
    return {"status": "timeout", "models_url": models_url, "reason": last_error, "log_tail": tail_text(log_path)}


def launch_vllm_server(config: VllmServerConfig) -> dict:
    vllm_bin = shutil.which("vllm")
    if not vllm_bin:
        return {"status": "blocked", "reason": "No encontre el comando vllm. Activa RUN_INSTALL=True o instala vllm."}

    log_path = "vllm_server.log"
    base_url = f"http://{config.host}:{config.port}/v1"
    existing = wait_for_models(base_url, timeout_s=3)
    if existing["status"] == "ok":
        return {"status": "already_running", "base_url": base_url, "health": existing}

    cleanup_gpu_and_vllm_processes()
    cmd = [
        vllm_bin,
        "serve",
        config.model,
        "--host",
        config.host,
        "--port",
        str(config.port),
        "--served-model-name",
        config.served_model_name,
        "--gpu-memory-utilization",
        str(config.gpu_memory_utilization),
        "--max-model-len",
        str(config.max_model_len),
        "--max-num-seqs",
        str(config.max_num_seqs),
    ]
    if config.enable_tool_choice:
        cmd += ["--enable-auto-tool-choice", "--tool-call-parser", config.tool_call_parser]
    if config.reasoning_parser:
        cmd += ["--reasoning-parser", config.reasoning_parser]

    process = subprocess.Popen(cmd, stdout=open(log_path, "w", encoding="utf-8"), stderr=subprocess.STDOUT)
    health = wait_for_models(base_url, process, log_path=log_path, timeout_s=300)
    if health["status"] == "ok":
        os.environ["VLLM_BASE_URL"] = base_url
        os.environ["VLLM_MODEL"] = config.served_model_name
        return {"status": "started", "pid": process.pid, "base_url": base_url, "cmd": cmd, "log_path": log_path, "health": health}
    return {"status": "starting_or_failed", "pid": process.pid, "returncode": process.poll(), "base_url": base_url, "cmd": cmd, "log_path": log_path, "health": health}


if RUN_VLLM_SERVER_SETUP:
    server_config = vllm_profile(VLLM_MODE, VLLM_MODEL)
    server_status = launch_vllm_server(server_config)
else:
    server_status = {"status": "skipped", "reason": "RUN_VLLM_SERVER_SETUP=False"}

toolkit.show(server_status, title="vLLM server launch opcional")


## 6) Health check del endpoint

Este es el gate real. Si `/models` no responde, no ejecutamos inferencia.


In [ ]:
health = wait_for_models(VLLM_BASE_URL, timeout_s=5)
VLLM_SERVER_AVAILABLE = health["status"] == "ok"
toolkit.show(health, title="vLLM server health")


## 7) Smoke directo con OpenAI SDK

Primero se prueba el endpoint sin Agentic Systems. En modelos Qwen3 con reasoning parser, vLLM puede separar la respuesta en `message.reasoning` y dejar `message.content=None`.

Para este smoke queremos validar texto final, no razonamiento interno; por eso la celda intenta desactivar thinking con `extra_body={"chat_template_kwargs": {"enable_thinking": False}}` cuando el backend lo soporta. Si aun asi no hay `content`, mostramos `reasoning` como diagnostico.


In [ ]:
if not RUN_OPENAI_SDK_SMOKE:
    toolkit.show({"status": "skipped", "reason": "RUN_OPENAI_SDK_SMOKE=False"}, title="OpenAI SDK smoke")
elif not VLLM_SERVER_AVAILABLE:
    toolkit.show({"status": "skipped", "reason": "Servidor vLLM no disponible."}, title="OpenAI SDK smoke")
else:
    from openai import OpenAI

    client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)
    request_kwargs = {
        "model": VLLM_MODEL,
        "messages": [
            {
                "role": "system",
                "content": "Responde directo, sin razonamiento, sin markdown y sin explicaciones.",
            },
            {
                "role": "user",
                "content": "Responde exactamente: hola mundo :)"
            },
        ],
        "temperature": 0.0,
        "max_tokens": 128,
    }
    if VLLM_DISABLE_THINKING_FOR_SDK_SMOKE:
        request_kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False}}

    response = client.chat.completions.create(**request_kwargs)
    message = response.choices[0].message
    message_payload = message.model_dump(mode="json")
    content = message_payload.get("content")
    reasoning = message_payload.get("reasoning") or message_payload.get("reasoning_content")
    text = content or reasoning
    status = "ok" if content else ("reasoning_only" if reasoning else "empty_content")

    toolkit.show(
        {
            "status": status,
            "text": text,
            "content": content,
            "reasoning": reasoning,
            "disable_thinking_requested": VLLM_DISABLE_THINKING_FOR_SDK_SMOKE,
            "message": message_payload,
            "usage": response.usage.model_dump(mode="json") if response.usage else None,
        },
        title="OpenAI SDK smoke",
    )


## 8) Agentic Systems smoke con `vllm-runtime`

Esta celda usa la misma forma de impresi?n que los dem?s tutorials: `human_result(...)` y `result.normalized()`.


In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos enteros."""
    return {"operation": "sumar", "result": a + b}

policy = toolkit.RunPolicy(
    max_turns=4,
    max_tool_calls=2,
    temperature=0.0,
    tool_choice="auto",
    repair=True,
    max_repairs=1,
    trace="compact",
    strict=True,
)

if not RUN_AGENTIC_SYSTEMS_SMOKE:
    toolkit.show({"status": "skipped", "reason": "RUN_AGENTIC_SYSTEMS_SMOKE=False"}, title="Tool smoke vLLM")
    result = None
elif not VLLM_SERVER_AVAILABLE:
    toolkit.show({"status": "skipped", "reason": "Servidor vLLM no disponible."}, title="Tool smoke vLLM")
    result = None
else:
    system = toolkit.AgenticSystem(runtime=vllm_runtime)
    agent = system.agent(
        name="vllm_calculator",
        instructions=(
            "Usa la tool sumar para resolver la suma solicitada. "
            "Al final responde en lenguaje natural, por ejemplo: 'La suma de 11 y 200 es 211.'"
        ),
        tools=[sumar],
    )
    result = agent.run("Suma 11 y 200 usando la tool sumar.", config=policy)
    toolkit.human_result(result, pretty=False, show_lineage=True)
    toolkit.show(result.normalized(), title="vLLM runtime normalized result")


## 9) Roadmap futuro: Unsloth -> fine-tuning -> export -> vLLM -> Agentic Systems

Este notebook cubre inferencia. La ruta futura queda documentada, no estabilizada como API:

```text
Fine-tuning / continued pretraining
    -> Unsloth
    -> export merged_16bit, LoRA o merged_4bit
    -> vLLM server OpenAI-compatible
    -> Agentic Systems runtime(provider="vllm-runtime")
    -> Skills / Agents / Systems / Evals
```


In [ ]:
api_coverage = [
    {"api": "toolkit.runtime(provider='vllm-runtime')", "description": "Declara vLLM como provider canonico OpenAI-compatible."},
    {"api": "toolkit.runtime(provider='auto')", "description": "Selecciona vLLM automaticamente cuando VLLM_BASE_URL esta configurado."},
    {"api": "RuntimeConfig.describe", "description": "Muestra resolucion y configuracion segura."},
    {"api": "toolkit.scheduler", "description": "Declara limites de ejecucion."},
    {"api": "toolkit.RunPolicy", "description": "Controla turns, tools, temperatura y reparacion."},
    {"api": "toolkit.tool", "description": "Define tools ejecutables por el agente."},
    {"api": "toolkit.AgenticSystem", "description": "Agrupa runtime, tools y agentes nativos."},
    {"api": "toolkit.human_result", "description": "Renderiza resultados humanos estables."},
]

toolkit.show({"notebook": "00_runtime_vllm_provider_api.ipynb", "api_coverage": api_coverage}, title="Cobertura API del notebook")
